# Pipeline IQ — Analysis Walkthrough

A guided tour of the B2B revenue analytics: warehouse → KPIs → segmentation → forecast → win drivers → recommendations. All heavy lifting lives in `src/`; this notebook narrates the results.

> Run `python src/run_all.py` once before this notebook so the data and warehouse exist.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.chdir(os.path.abspath('..'))  # run from project root so paths resolve

import pandas as pd
from etl import build_warehouse, table, q

con = build_warehouse()
print('warehouse ready — fct_pipeline rows:', con.execute('SELECT COUNT(*) FROM fct_pipeline').fetchone()[0])

## 1. The data model
The SQL layer produces a star schema. `fct_pipeline` is one row per opportunity, enriched with account, product, and sales-team attributes.

In [ ]:
table(con, 'fct_pipeline').head()

## 2. Headline KPIs
Win rate, cycle time, and revenue — sliced by sector.

In [ ]:
seg = table(con, 'kpi_segment')
seg[['sector','opportunities','win_rate','revenue','avg_cycle_days']]

In [ ]:
# Rep leaderboard (top 10 by revenue)
table(con, 'kpi_agent').head(10)[['sales_agent','regional_office','won_deals','win_rate','revenue']]

## 3. Account segmentation (RFM + K-means)
We adapt RFM to B2B (recency of last opportunity, opportunity frequency, total won revenue) and cluster into business tiers.

In [ ]:
from segmentation import segment_accounts
seg_acc = segment_accounts(con)
seg_acc.groupby('tier').agg(accounts=('account','count'),
                            avg_rfm=('rfm_score','mean'),
                            total_won=('monetary','sum')).sort_values('avg_rfm', ascending=False)

## 4. Backtested revenue forecast
Two transparent methods are compared with a walk-forward backtest; the lower-MAPE method drives the forecast.

In [ ]:
from forecast import run as run_forecast
fc = run_forecast(con)
print('method MAPEs:', {k: round(v,3) for k,v in fc['method_scores'].items()})
print('selected     :', fc['best_method'])
print('next quarter :', f"${fc['next_quarter_statistical']:,.0f}")
fc['forecast']

## 5. What drives a win?
A logistic model over closed deals ranks the drivers of win probability (odds ratios > 1 raise win odds).

In [ ]:
from drivers import fit_driver_model
dr = fit_driver_model(con)
print('cross-validated ROC-AUC:', round(dr['auc'], 3))
dr['drivers'].head(10)

## 6. Recommendations
See [`reports/recommendation_memo.md`](../reports/recommendation_memo.md) for the full write-up:

1. **Reallocate** prospecting from low-converting sectors (Marketing/Entertainment) to Technology/Software.
2. **Prioritize** Enterprise-tier accounts — they convert materially better.
3. **Protect** the ~20% of accounts driving ~27% of revenue with named coverage.
4. **Re-forecast** monthly and track MAPE to keep quota defensible.